# 03 — Lightning Session Comparison

Weekly inter-rater reliability check: all digitizers map the **same randomly assigned cell**.
Collect their `.geojson` files in `data/processed/lightning/` then run this notebook.

**What it does:**
1. Loads the road-intersect grid for the settlement and locates the target cell.
2. Reads every `.geojson` in `data/processed/lightning/` — one file per digitizer.
3. Computes per-digitizer geometry metrics (length, node density, segment count).
4. Computes pairwise F-measure (precision / recall / F1) and buffered IoU for every pair.
5. Produces an overlay map of all digitizers on the single cell.
6. Saves figures and tables to `outputs/lightning/{settlement}/`.

**File naming for lightning digitizations:**
Place one `.geojson` per digitizer in `data/processed/lightning/`.  
The digitizer label is taken from the filename stem (e.g. `casey.geojson` → *Casey*).

In [ ]:
# ── Parameters — edit this cell only ─────────────────────────────────────────

SETTLEMENT   = "nakivale"   # must match <settlement>_1km_road_intersect_1km.geojson
LIGHTNING_OID = None         # None = pick randomly; set to an integer to pin the cell

PROJECTED_CRS        = "EPSG:32636"  # UTM Zone 36N
CELL_SIZE_M          = 1000
SNAP_BUFFER_M        = 5             # buffer radius (m) for buffered IoU
FMEASURE_THRESHOLD_M = 10            # distance threshold (m) for F-measure
FMEASURE_SAMPLE_M    = 2             # point-sampling interval (m) along lines

RANDOM_SEED = None   # integer seed for reproducible random cell selection, or None

In [ ]:
import warnings
from itertools import combinations
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from scipy import stats
from scipy.spatial import cKDTree
from shapely.ops import unary_union

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

In [ ]:
def find_project_root(start=None):
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "configs" / "paths.yaml").exists():
            return candidate
    raise FileNotFoundError("configs/paths.yaml not found")

project_root = find_project_root()
with open(project_root / "configs" / "paths.yaml") as f:
    _paths = yaml.safe_load(f)

processed_dir  = project_root / _paths["data"]["processed"]
lightning_dir  = processed_dir / "lightning"

out_dir = project_root / "outputs" / "lightning" / SETTLEMENT
out_dir.mkdir(parents=True, exist_ok=True)

# Colour palette — up to 8 digitizers
_PALETTE = [
    "#2166ac", "#d6604d", "#1a9641", "#762a83",
    "#e08214", "#4dac26", "#d01c8b", "#313695",
]

print(f"Settlement    : {SETTLEMENT}")
print(f"Lightning dir : {lightning_dir}")
print(f"Output dir    : {out_dir}")

def savefig(name: str, fig=None, **kwargs):
    path = out_dir / f"{name}.png"
    (fig or plt).savefig(path, dpi=150, bbox_inches="tight", **kwargs)
    print(f"  Saved → {path.relative_to(project_root)}")

def savecsv(df: pd.DataFrame, name: str):
    path = out_dir / f"{name}.csv"
    df.to_csv(path, index=True)
    print(f"  Saved → {path.relative_to(project_root)}")

## 1  Load grid and select lightning cell

In [ ]:
grid_path = processed_dir / f"{SETTLEMENT}_1km_road_intersect_1km.geojson"
if not grid_path.exists():
    raise FileNotFoundError(
        f"Grid not found: {grid_path}\n"
        f"Expected: data/processed/{SETTLEMENT}_1km_road_intersect_1km.geojson"
    )

grid = gpd.read_file(grid_path).to_crs(PROJECTED_CRS)
print(f"Grid loaded: {len(grid)} cells  CRS: {grid.crs.to_epsg()}")
print(f"Columns: {list(grid.columns)}")

rng = np.random.default_rng(RANDOM_SEED)
if LIGHTNING_OID is None:
    oid = int(rng.choice(grid["OID"].values))
    print(f"\nRandomly selected OID: {oid}  (set LIGHTNING_OID={oid} to pin this cell)")
else:
    oid = int(LIGHTNING_OID)
    print(f"\nUsing pinned OID: {oid}")

cell_row  = grid[grid["OID"] == oid].iloc[0]
cell_geom = cell_row.geometry
print(f"Cell area: {cell_geom.area / 1e6:.3f} km²")

## 2  Load digitizer files

In [ ]:
dig_files = sorted(lightning_dir.glob("*.geojson"))
if not dig_files:
    raise FileNotFoundError(
        f"No .geojson files found in {lightning_dir}\n"
        "Place one file per digitizer there and re-run."
    )

LABELS = [f.stem.capitalize() for f in dig_files]
COLORS = {label: _PALETTE[i % len(_PALETTE)] for i, label in enumerate(LABELS)}
pairs  = list(combinations(LABELS, 2))

lines_gdfs = {}
for label, path in zip(LABELS, dig_files):
    raw = gpd.read_file(path).to_crs(PROJECTED_CRS)
    lines_gdfs[label] = raw.explode(index_parts=False).reset_index(drop=True)
    print(f"  {label:15s}: {len(lines_gdfs[label])} features  ({path.name})")

print(f"\n{len(LABELS)} digitizers: {LABELS}")
print(f"{len(pairs)} pairwise comparisons")

## 3  Geometry helpers

In [ ]:
CELL_AREA_KM2 = (CELL_SIZE_M / 1000) ** 2

def _count_nodes(geom) -> int:
    if geom is None or geom.is_empty:
        return 0
    t = geom.geom_type
    if t in ("LineString", "LinearRing"):
        return len(geom.coords)
    if t == "Point":
        return 1
    if t in ("MultiLineString", "MultiPoint", "MultiPolygon", "GeometryCollection"):
        return sum(_count_nodes(part) for part in geom.geoms)
    if t == "Polygon":
        return len(geom.exterior.coords) + sum(len(r.coords) for r in geom.interiors)
    return 0

def _extract_lines(geom):
    if geom is None or geom.is_empty:
        return []
    t = geom.geom_type
    if t in ("LineString", "LinearRing"):
        return [geom]
    if t == "MultiLineString":
        return list(geom.geoms)
    if t in ("GeometryCollection", "MultiPolygon", "MultiPoint"):
        out = []
        for part in geom.geoms:
            out.extend(_extract_lines(part))
        return out
    return []

def _internode_distances(line) -> np.ndarray:
    coords = np.array(line.coords)
    if len(coords) < 2:
        return np.array([])
    return np.sqrt(((coords[1:] - coords[:-1]) ** 2).sum(axis=1))

def _sample_points_along_lines(lines_gdf, cell_geom, interval_m):
    sub = lines_gdf[lines_gdf.intersects(cell_geom)].copy()
    if sub.empty:
        return None
    sub["geometry"] = sub.geometry.intersection(cell_geom)
    sub = sub[~sub.geometry.is_empty]
    if sub.empty:
        return None
    pts = []
    for geom in sub.geometry:
        for line in _extract_lines(geom):
            if line.length == 0:
                continue
            for d in np.arange(0, line.length, interval_m):
                p = line.interpolate(d)
                pts.append((p.x, p.y))
    return np.array(pts) if pts else None

def metrics_for_lines_in_cell(lines_gdf, cell_geom) -> dict:
    _empty = dict(
        n_segments=0, n_nodes=0,
        total_length_m=0.0, density_m_per_km2=0.0,
        mean_seg_length_m=float("nan"),
        nodes_per_km=float("nan"),
        mean_internode_dist_m=float("nan"),
    )
    clipped = lines_gdf[lines_gdf.intersects(cell_geom)].copy()
    if clipped.empty:
        return _empty
    clipped["geometry"] = clipped.geometry.intersection(cell_geom)
    clipped = clipped[~clipped.geometry.is_empty].copy()
    if clipped.empty:
        return _empty
    all_lines = []
    for geom in clipped.geometry:
        all_lines.extend(_extract_lines(geom))
    if not all_lines:
        return _empty
    lengths = np.array([g.length for g in all_lines])
    total   = lengths.sum()
    n_nodes = int(sum(_count_nodes(g) for g in all_lines))
    all_gaps = np.concatenate([_internode_distances(ln) for ln in all_lines])
    mean_gap = float(all_gaps.mean()) if len(all_gaps) > 0 else float("nan")
    return dict(
        n_segments            = len(all_lines),
        n_nodes               = n_nodes,
        total_length_m        = total,
        density_m_per_km2     = total / CELL_AREA_KM2,
        mean_seg_length_m     = float(lengths.mean()),
        nodes_per_km          = n_nodes / (total / 1000) if total > 0 else float("nan"),
        mean_internode_dist_m = mean_gap,
    )

def line_fmeasure(lines_A, lines_B, cell_geom, threshold_m, interval_m):
    pts_a = _sample_points_along_lines(lines_A, cell_geom, interval_m)
    pts_b = _sample_points_along_lines(lines_B, cell_geom, interval_m)
    if pts_a is None or pts_b is None:
        return dict(precision=np.nan, recall=np.nan, f1=np.nan)
    tree_b = cKDTree(pts_b)
    tree_a = cKDTree(pts_a)
    dist_ab, _ = tree_b.query(pts_a, workers=-1)
    dist_ba, _ = tree_a.query(pts_b, workers=-1)
    precision = float((dist_ab <= threshold_m).mean())
    recall    = float((dist_ba <= threshold_m).mean())
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return dict(precision=precision, recall=recall, f1=f1)

def _clipped_buffer_union(lines_gdf, cell_geom, buf_m):
    sub = lines_gdf[lines_gdf.intersects(cell_geom)].copy()
    if sub.empty:
        return None
    sub["geometry"] = sub.geometry.intersection(cell_geom)
    sub = sub[~sub.geometry.is_empty]
    return unary_union(sub.geometry.buffer(buf_m)) if not sub.empty else None

def pairwise_iou(lA, lB, cell_geom, buf_m):
    pA = _clipped_buffer_union(lines_gdfs[lA], cell_geom, buf_m)
    pB = _clipped_buffer_union(lines_gdfs[lB], cell_geom, buf_m)
    if pA is None and pB is None:
        return np.nan
    if pA is None or pB is None:
        return 0.0
    u = pA.union(pB).area
    return pA.intersection(pB).area / u if u > 0 else np.nan

print("Geometry helpers defined.")

## 4  Per-digitizer metrics on the lightning cell

In [ ]:
AGG_COLS = [
    "total_length_m", "density_m_per_km2",
    "n_segments", "n_nodes",
    "nodes_per_km", "mean_internode_dist_m",
    "mean_seg_length_m",
]

metric_rows = []
for label in LABELS:
    row = {"digitizer": label, **metrics_for_lines_in_cell(lines_gdfs[label], cell_geom)}
    metric_rows.append(row)

metrics_df = pd.DataFrame(metric_rows).set_index("digitizer")

print(f"=== Per-digitizer metrics — OID {oid} ===")
display(metrics_df[AGG_COLS].round(2))
savecsv(metrics_df[AGG_COLS], f"01_metrics_oid{oid}")

## 5  Metric bar charts — all digitizers

In [ ]:
fig, axes = plt.subplots(1, len(AGG_COLS), figsize=(22, 4))
fig.suptitle(f"Lightning session — OID {oid} ({SETTLEMENT})  |  per-digitizer metrics",
             fontsize=11)
x_pos = np.arange(len(LABELS))
for ax, col in zip(axes, AGG_COLS):
    vals  = [metrics_df.loc[lbl, col] if lbl in metrics_df.index else 0 for lbl in LABELS]
    colors = [COLORS[lbl] for lbl in LABELS]
    bars  = ax.bar(x_pos, vals, color=colors, edgecolor="white", alpha=0.85)
    ax.bar_label(bars, fmt="%.0f", fontsize=7, padding=2)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(LABELS, fontsize=8, rotation=30, ha="right")
    ax.set_title(col.replace("_", "\n"), fontsize=8)
    ax.set_ylim(0, max(vals + [1]) * 1.25)
    ax.tick_params(labelsize=8)
plt.tight_layout()
savefig(f"02_metric_bars_oid{oid}")
plt.show()

## 6  Overlay map — all digitizers on the lightning cell

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
gpd.GeoSeries([cell_geom]).boundary.plot(ax=ax, color="black", linewidth=2.0, zorder=1)

for label in LABELS:
    clipped = lines_gdfs[label][lines_gdfs[label].intersects(cell_geom)].copy()
    if not clipped.empty:
        clipped["geometry"] = clipped.geometry.intersection(cell_geom)
        clipped = clipped[~clipped.geometry.is_empty]
    if not clipped.empty:
        m = metrics_df.loc[label]
        clipped.plot(
            ax=ax, color=COLORS[label], linewidth=1.8, zorder=3,
            label=f"{label}  ({m.total_length_m:.0f} m, {m.nodes_per_km:.1f} nodes/km)"
        )

ax.set_title(
    f"Lightning session — OID {oid} ({SETTLEMENT})\n"
    f"{len(LABELS)} digitizers",
    fontsize=11,
)
ax.legend(fontsize=8, loc="upper right")
ax.set_axis_off()
plt.tight_layout()
savefig(f"03_overlay_oid{oid}")
plt.show()

## 7  Individual digitizer panels

In [ ]:
n_cols = min(4, len(LABELS))
n_rows = int(np.ceil(len(LABELS) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols,
                          figsize=(n_cols * 4, n_rows * 4),
                          squeeze=False)
fig.suptitle(f"Lightning session — OID {oid} ({SETTLEMENT})  |  individual panels",
             fontsize=11)

for idx, label in enumerate(LABELS):
    ax = axes[idx // n_cols][idx % n_cols]
    gpd.GeoSeries([cell_geom]).boundary.plot(ax=ax, color="black", linewidth=1.5, zorder=1)
    clipped = lines_gdfs[label][lines_gdfs[label].intersects(cell_geom)].copy()
    if not clipped.empty:
        clipped["geometry"] = clipped.geometry.intersection(cell_geom)
        clipped = clipped[~clipped.geometry.is_empty]
    if not clipped.empty:
        clipped.plot(ax=ax, color=COLORS[label], linewidth=1.2, zorder=2)
    m = metrics_df.loc[label]
    ax.set_title(
        f"{label}\n"
        f"{m.total_length_m:.0f} m | {m.n_segments:.0f} segs | "
        f"{m.nodes_per_km:.1f} nodes/km",
        fontsize=9,
    )
    ax.set_axis_off()

# hide any unused subplot panels
for idx in range(len(LABELS), n_rows * n_cols):
    axes[idx // n_cols][idx % n_cols].set_visible(False)

plt.tight_layout()
savefig(f"04_individual_panels_oid{oid}")
plt.show()

## 8  Pairwise agreement — F-measure & buffered IoU

**F-measure at threshold T:**
- Sample points every `FMEASURE_SAMPLE_M` m along A's lines
- **Precision** = fraction of A's points within T m of any line in B
- **Recall** = fraction of B's points within T m of any line in A
- **F1** = harmonic mean

Low precision → A drew roads B missed. Low recall → B drew roads A missed.

In [ ]:
pair_rows = []
for (lA, lB) in pairs:
    fm  = line_fmeasure(lines_gdfs[lA], lines_gdfs[lB],
                        cell_geom, FMEASURE_THRESHOLD_M, FMEASURE_SAMPLE_M)
    iou = pairwise_iou(lA, lB, cell_geom, SNAP_BUFFER_M)
    pair_rows.append({
        "pair": f"{lA} vs {lB}",
        "digitizer_A": lA,
        "digitizer_B": lB,
        "precision":   fm["precision"],
        "recall":      fm["recall"],
        "f1":          fm["f1"],
        "iou":         iou,
    })

pair_df = pd.DataFrame(pair_rows).set_index("pair")
print(f"=== Pairwise F-measure (T={FMEASURE_THRESHOLD_M} m) & IoU (buf={SNAP_BUFFER_M} m) ===")
display(pair_df.drop(columns=["digitizer_A", "digitizer_B"]).round(3))
savecsv(pair_df, f"05_pairwise_agreement_oid{oid}")

## 9  Pairwise agreement matrix (F1 heatmap)

In [ ]:
# Build symmetric F1 matrix
f1_matrix = pd.DataFrame(np.nan, index=LABELS, columns=LABELS)
iou_matrix = pd.DataFrame(np.nan, index=LABELS, columns=LABELS)
for _, row in pair_df.iterrows():
    lA, lB = row["digitizer_A"], row["digitizer_B"]
    f1_matrix.loc[lA, lB] = row["f1"]
    f1_matrix.loc[lB, lA] = row["f1"]
    iou_matrix.loc[lA, lB] = row["iou"]
    iou_matrix.loc[lB, lA] = row["iou"]
np.fill_diagonal(f1_matrix.values, 1.0)
np.fill_diagonal(iou_matrix.values, 1.0)

fig, axes = plt.subplots(1, 2, figsize=(max(8, len(LABELS) * 1.6) * 2, max(6, len(LABELS) * 1.6)))
for ax, mat, title in zip(axes,
                           [f1_matrix, iou_matrix],
                           [f"F1 (T={FMEASURE_THRESHOLD_M} m)", f"Buffered IoU (buf={SNAP_BUFFER_M} m)"]):
    im = ax.imshow(mat.values.astype(float), vmin=0, vmax=1, cmap="RdYlGn", aspect="auto")
    ax.set_xticks(range(len(LABELS)))
    ax.set_yticks(range(len(LABELS)))
    ax.set_xticklabels(LABELS, rotation=45, ha="right", fontsize=9)
    ax.set_yticklabels(LABELS, fontsize=9)
    for i in range(len(LABELS)):
        for j in range(len(LABELS)):
            v = mat.values[i, j]
            if not np.isnan(v):
                ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                        fontsize=9, color="black" if 0.35 < v < 0.85 else "white")
    ax.set_title(title, fontsize=10)
    plt.colorbar(im, ax=ax, shrink=0.8)

fig.suptitle(f"Lightning session — OID {oid} ({SETTLEMENT})  |  pairwise agreement",
             fontsize=11)
plt.tight_layout()
savefig(f"06_f1_iou_heatmap_oid{oid}")
plt.show()

## 10  Pairwise side-by-side maps

In [ ]:
def _plot_digitizer_on_cell(ax, label, cell_geom):
    gpd.GeoSeries([cell_geom]).boundary.plot(ax=ax, color="black", linewidth=1.5, zorder=1)
    clipped = lines_gdfs[label][lines_gdfs[label].intersects(cell_geom)].copy()
    if not clipped.empty:
        clipped["geometry"] = clipped.geometry.intersection(cell_geom)
        clipped = clipped[~clipped.geometry.is_empty]
    if not clipped.empty:
        clipped.plot(ax=ax, color=COLORS[label], linewidth=1.2, zorder=2)
    m = metrics_df.loc[label]
    ax.set_title(
        f"{label}\n{m.total_length_m:.0f} m | {m.n_segments:.0f} segs | "
        f"{m.nodes_per_km:.1f} nodes/km",
        fontsize=8,
    )
    ax.set_axis_off()

for (lA, lB) in pairs:
    row = pair_df.loc[f"{lA} vs {lB}"]
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    fig.suptitle(
        f"{lA} vs {lB} — OID {oid}  |  "
        f"P={row.precision:.2f}  R={row.recall:.2f}  F1={row.f1:.2f}  "
        f"IoU={row.iou:.2f}",
        fontsize=10,
    )
    _plot_digitizer_on_cell(axes[0], lA, cell_geom)
    _plot_digitizer_on_cell(axes[1], lB, cell_geom)
    plt.tight_layout()
    savefig(f"07_pair_{lA.lower()}_{lB.lower()}_oid{oid}")
    plt.show()

## 11  Precision vs Recall scatter — all pairs

In [ ]:
if len(pairs) > 1:
    fig, ax = plt.subplots(figsize=(6, 6))
    for (lA, lB) in pairs:
        row = pair_df.loc[f"{lA} vs {lB}"]
        ax.scatter(row.precision, row.recall, s=80, zorder=3,
                   label=f"{lA} vs {lB}  (F1={row.f1:.2f})")
        ax.annotate(
            f"{lA[:3]}/{lB[:3]}",
            (row.precision, row.recall),
            fontsize=7, xytext=(4, 3), textcoords="offset points",
        )
    ax.plot([0, 1], [0, 1], "--", color="#cccccc", linewidth=1)
    ax.set_xlim(-0.02, 1.05)
    ax.set_ylim(-0.02, 1.05)
    ax.set_xlabel(f"Precision (A's pts within {FMEASURE_THRESHOLD_M} m of B)", fontsize=9)
    ax.set_ylabel(f"Recall (B's pts within {FMEASURE_THRESHOLD_M} m of A)", fontsize=9)
    ax.set_title(
        f"Precision vs Recall — all pairs  |  OID {oid}",
        fontsize=10,
    )
    ax.legend(fontsize=7, loc="lower right")
    plt.tight_layout()
    savefig(f"08_prec_recall_scatter_oid{oid}")
    plt.show()
else:
    print("Only one pair — scatter plot skipped.")

## 12  Summary table

In [ ]:
print(f"\n{'='*60}")
print(f"Lightning session summary — OID {oid} ({SETTLEMENT})")
print(f"{'='*60}")
print(f"  Digitizers : {', '.join(LABELS)}")
print(f"  Pairs      : {len(pairs)}")
print()

print("Per-digitizer metrics:")
display(metrics_df[AGG_COLS].round(2))

print("\nPairwise agreement:")
display(pair_df.drop(columns=["digitizer_A","digitizer_B"]).round(3))

if len(pairs) > 0:
    mean_f1  = pair_df["f1"].mean()
    mean_iou = pair_df["iou"].mean()
    print(f"\nMean F1 across all pairs : {mean_f1:.3f}")
    print(f"Mean IoU across all pairs: {mean_iou:.3f}")

print(f"\nOutputs written to: outputs/lightning/{SETTLEMENT}/")